In [22]:
!pip install gensim nltk scikit-learn

In [23]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

from nltk.tokenize import word_tokenize
import numpy as np
from gensim.models import Word2Vec, FastText
from sklearn.metrics.pairwise import cosine_similarity

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [24]:
def preprocess(text):
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word.isalnum()]
    return tokens

In [25]:
sentences = [
    "Best laptop for coding",
    "Top programming laptops for developers",
    "Affordable smartphones with good battery",
    "Best phones for gaming",
    "How to reset password",
    "Steps to change account password"
]

In [26]:
def train_word2vec(sentences):
    data = [preprocess(s) for s in sentences]
    model = Word2Vec(data, vector_size=100, window=5, min_count=1)
    return model

In [27]:
def train_fasttext(sentences):
    data = [preprocess(s) for s in sentences]
    model = FastText(data, vector_size=100, window=5, min_count=1)
    return model

In [28]:
def load_glove(file_path):
    glove_dict = {}

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            vector = np.asarray(values[1:], dtype='float32')
            glove_dict[word] = vector

    return glove_dict

In [29]:
def get_sentence_vector(model, sentence, model_type="word2vec"):
    words = preprocess(sentence)
    vectors = []

    for word in words:
        if model_type in ["word2vec", "fasttext"]:
            if word in model.wv:
                vectors.append(model.wv[word])
        elif model_type == "glove":
            if word in model:
                vectors.append(model[word])

    if len(vectors) == 0:
        return np.zeros(100)

    return np.mean(vectors, axis=0)

In [30]:
def semantic_search(query, sentences, model, model_type):
    query_vec = get_sentence_vector(model, query, model_type)

    scores = []

    for sentence in sentences:
        sent_vec = get_sentence_vector(model, sentence, model_type)

        sim = cosine_similarity([query_vec], [sent_vec])[0][0]
        scores.append(sim)

    best_index = np.argmax(scores)

    return sentences[best_index], scores[best_index]

In [31]:
# Train models
w2v_model = train_word2vec(sentences)
ft_model = train_fasttext(sentences)

# Load GloVe (upload file first)
glove_model = load_glove("glove.6B.100d.txt")

while True:
    query = input("Enter query (or 'exit'): ")

    if query.lower() == "exit":
        break

    print("\n--- Word2Vec Result ---")
    res, score = semantic_search(query, sentences, w2v_model, "word2vec")
    print(res, "| Score:", score)

    print("\n--- FastText Result ---")
    res, score = semantic_search(query, sentences, ft_model, "fasttext")
    print(res, "| Score:", score)

    print("\n--- GloVe Result ---")
    res, score = semantic_search(query, sentences, glove_model, "glove")
    print(res, "| Score:", score)

    print("\n===========================\n")


Enter query (or 'exit'): I forgot my password

--- Word2Vec Result ---
How to reset password | Score: 0.46209568

--- FastText Result ---
Steps to change account password | Score: 0.18364234

--- GloVe Result ---
How to reset password | Score: 0.7795752


Enter query (or 'exit'): password

--- Word2Vec Result ---
How to reset password | Score: 0.46209568

--- FastText Result ---
Affordable smartphones with good battery | Score: 0.2918975

--- GloVe Result ---
How to reset password | Score: 0.64459443


Enter query (or 'exit'): exit
